# 第 7 章：構建聊天應用程式
## Microsoft Foundry 模型 API 快速入門

此筆記本改編自包括存取 [Azure OpenAI](notebook-azure-openai.ipynb) 服務的 [Azure OpenAI 範例庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) 。

> **注意：** GitHub Models 將於 2026 年 7 月底退役。此筆記本現已針對提供相同免費試用模型目錄及 Azure AI 推論 SDK 體驗的 [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)。


# 概覽  
「大型語言模型是將文字映射到文字的函數。給定一個輸入字串，大型語言模型會嘗試預測接下來會出現的文字」（1）。這份「快速入門」筆記本將向使用者介紹大型語言模型的高階概念、開始使用 AML 所需的核心套件、提示設計的簡單介紹，以及幾個不同使用案例的短示範。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 創建你的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 憑證](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文本](#1.-summarize-text)  
[2. 對文本進行分類](#2.-classify-text)  
[3. 生成新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立你的第一個提示  
這個簡短練習將為你介紹如何向 Microsoft Foundry Models 提交提示以執行簡單任務「摘要」。  


<strong>步驟</strong>：  
1. 如果還未安裝，請在你的 python 環境中安裝 `azure-ai-inference`函式庫。  
2. 載入標準輔助函式庫並設定你的 Microsoft Foundry Models 憑證。  
3. 為你的任務選擇一個模型  
4. 為模型建立一個簡單的提示  
5. 向模型 API 提交你的請求！  


### 1. 安裝 `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. 導入輔助庫並實例化憑證


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. 搵啱嘅模型  
好似 GPT-4o 同 GPT-4o mini 呢啲模型可以理解同生成自然語言，喺 Microsoft Foundry Models 目錄入面，可以搵到佢哋同 Meta、Mistral、Cohere 同 Microsoft 嘅模型。 


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. 提示設計  

「大型語言模型的魔力在於，通過在大量文本上訓練以最小化這種預測錯誤，模型最終學習到了對這些預測有用的概念。例如，它們學會了這樣的概念」(1):

* 如何拼寫
* 語法如何運作
* 如何改寫
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編程
* 等等。

#### 如何控制大型語言模型  
「在所有輸入給大型語言模型的項目中，最具影響力的當然是文本提示」(1)。

大型語言模型可以通過幾種方式被提示以生成輸出：

指令：告訴模型你想要什麼
補全：引導模型補全你想要的開頭部分
示範：通過以下任一方式向模型展示你想要的：
在提示中提供幾個例子
在微調訓練數據集中提供數百或數千個例子」



#### 建立提示有三個基本準則：

<strong>展示與說明</strong>。透過指令、範例或兩者結合讓它明確知道你想要什麼。如果你想讓模型按字母順序排列項目清單或依情感對段落分類，就向它展示這正是你想要的。

<strong>提供優質資料</strong>。如果你試圖建立一個分類器或希望模型遵循某模式，請確保範例充足。務必校對你的範例——模型通常足夠聰明能識別基本拼寫錯誤並給出回應，但它也可能會認為這是故意的，並且這可能會影響回應。

**檢查設定。** 溫度和 top_p 設定控制模型生成回答的確定性。若你要求模型給出只有一個正確答案的回應，則應將這些數值調低。若你想要更多元的回應，則可調高它們。人們對這些設定最大的誤解是以為它們是「聰明度」或「創造力」的控制。


來源： https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### 重複相同的呼叫，結果有何不同？ 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## 摘要文本  
#### 挑戰  
透過在段落末尾加入「tl;dr:」來對文本進行摘要。留意模型如何在沒有額外指示的情況下理解並執行多種任務。你可以嘗試比 tl;dr 更具描述性的提示詞，以改變模型的行為並自訂你收到的摘要(3)。  

最近的研究證明，先在大量文本語料上進行預訓練，然後再針對特定任務進行微調，在許多自然語言處理任務和基準測試中取得了顯著提升。儘管此方法的架構通常不依賴特定任務，但仍需成千上萬的任務特定微調數據集。相比之下，人類通常只需少量範例或簡單指示便能執行新語言任務——而目前的自然語言處理系統仍在這方面存在較大挑戰。本文展示了擴大量語言模型規模能大幅提升與任務無關的少量示例表現，有時甚至能達到先前最先進微調方法的競爭力。 



摘要  


# 多種使用場景練習  
1. 摘要文本  
2. 分類文本  
3. 生成新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 分類文字  
#### 挑戰  
在推理階段將項目分類到提供的類別中。在以下範例中，我們在提示中同時提供類別和要分類的文字(*playground_reference)。 

客戶查詢：您好，我的筆記型電腦鍵盤最近有一個鍵壞了，我需要更換：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 產生新產品名稱
#### 挑戰
從範例詞彙建立產品名稱。在提示中我們包括有關我們要產生名稱的產品資訊。我們也提供相似的範例來展示我們希望得到的模式。我們也把溫度參數設定為較高，以增加隨機性和更具創新性的回應。

產品描述：一台家用奶昔機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙可適合任何腳尺寸的鞋
種子詞：適應性、合腳、全尺寸合腳。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry 入口網站](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微調 GPT 模型以分類文本的最佳實踐](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 獲取更多幫助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
